In [8]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
load_dotenv(override=True)


True

In [ ]:
#Ticker is a unique identifier of US Companies e.g. AAPL for Apple Inc.
def get_cik(ticker:str) -> str:
    url= "https://www.sec.gov/files/company_tickers.json"
    headers={'User-Agent': os.getenv('SEC_USER_AGENT')}
    response=requests.get(url, headers=headers)
    response.raise_for_status()
    data=response.json()
    
    #Search for ticker
    for company in data.values():
        if company['ticker'].upper() ==ticker.upper():
            cik=str(company['cik_str']).zfill(10)
            return cik
        
    return None

In [10]:
#Test
cik = get_cik('AAPL')
print(f"Apple CIK: {cik}")

Apple CIK: 0000320193


In [ ]:
def get_financial_data(ticker):
    cik=get_cik(ticker)
    if not cik:
        print(f"No cik found for {ticker}")
        return None
    

    print(f"Found CIK:{cik}")

    # Get company facts
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    headers = {'User-Agent': os.getenv('SEC_USER_AGENT')}

    print("Fetching data from SEC ... ")

    response=requests.get(url, headers=headers)

    #Checking Connection
    if response.status_code != 200:
        print(f"ERROR: Status Code: {response.status_code}")
        return None
    
    print(f"Data Recived!")

    data=response.json()

    #US GAAP=Generally Accepted Accounting Principles
    us_gaap=data['facts']['us-gaap']



    ##### BUILD YOUR OWN FUNCTION TO GET THE DATA YOU NEED
    ##### YOU NEED THE FOLLOWING:
    #### Revenue
    #    Net Income
    #    Total Assets
    #    Current Assets
    #    Current Liabilities
    #    Total Debt
    #    Shareholders Equity
    #    DATE


In [ ]:


    #Helper function to try multiple tags
    def get_value(tag_list):
        """Try multiple XBRL tags, return first match"""
        for tag in tag_list:
            if tag in us_gaap:
                records=us_gaap[tag]['units']['USD']
                annual=[r for r in records if r.get('form')=='10-K']
                if annual:
                    latest=sorted(annual, key=lambda x:x['end'], reverse=True)[0]
                    return latest['val'], latest['end']
        return None
    

    # Extract key metrics
    financial_data = {
        'ticker': ticker,
        'company_name': data.get('entityName'),
        'revenue': get_value([
            'RevenueFromContractWithCustomerExcludingAssessedTax',
            'Revenues',
            'SalesRevenueNet'
        ]),
        'net_income': get_value(['NetIncomeLoss', 'ProfitLoss']),
        'total_assets': get_value(['Assets']),
        'current_assets': get_value(['AssetsCurrent']),
        'current_liabilities': get_value(['LiabilitiesCurrent']),
        'total_debt': get_value(['LongTermDebt', 'DebtLongTerm']),
        'shareholders_equity': get_value(['StockholdersEquity', 'ShareholdersEquity'])
    }
    
    return financial_data
    




### IGNORE THIS!

In [28]:
# Cell: Better get_value function with validation

def get_value_with_validation(us_gaap, tag_list, metric_name):
    """
    Try multiple XBRL tags with extensive validation
    
    Returns: dict with value and metadata, or None
    """
    
    for tag in tag_list:
        if tag not in us_gaap:
            continue  # Tag doesn't exist, try next
        
        try:
            records = us_gaap[tag]['units']['USD']
        except KeyError:
            continue  # No USD units, try next tag
        
        # Filter for annual reports (10-K)
        annual = [r for r in records if r.get('form') == '10-K']
        
        if not annual:
            continue  # No 10-K filings, try next tag
        
        # Sort by end date (most recent first)
        sorted_annual = sorted(annual, key=lambda x: x['end'], reverse=True)
        
        # Get most recent
        latest = sorted_annual[0]
        
        # Validation checks
        if latest['val'] is None:
            continue  # Value is null, try next tag
        
        if latest['val'] == 0 and metric_name not in ['net_income']:  # Net income can be zero/negative
            continue  # Suspicious zero value, try next tag
        
        # Return with metadata for verification
        return {
            'value': latest['val'],
            'date': latest['end'],
            'tag_used': tag,
            'filed_date': latest.get('filed'),
            'fiscal_year': latest.get('fy'),
            'fiscal_period': latest.get('fp')
        }
    
    # None of the tags worked
    return None


# Test the improved function
def get_financial_data_validated(ticker):
    """Extract financial statements with validation"""
    
    # Get CIK
    cik = get_cik(ticker)
    if not cik:
        print(f"❌ Could not find CIK for {ticker}")
        return None
    
    print(f"✅ Found CIK: {cik}")
    
    # Get company facts
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    headers = {'User-Agent': os.getenv('SEC_USER_AGENT')}
    
    print(f"📡 Fetching data from SEC...")
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"❌ Error: API returned status {response.status_code}")
        return None
    
    data = response.json()
    us_gaap = data['facts']['us-gaap']
    
    # Extract with validation
    revenue_data = get_value_with_validation(
        us_gaap,
        ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues', 'SalesRevenueNet'],
        'revenue'
    )
    
    net_income_data = get_value_with_validation(
        us_gaap,
        ['NetIncomeLoss', 'ProfitLoss'],
        'net_income'
    )
    
    total_assets_data = get_value_with_validation(
        us_gaap,
        ['Assets'],
        'total_assets'
    )
    
    current_assets_data = get_value_with_validation(
        us_gaap,
        ['AssetsCurrent'],
        'current_assets'
    )
    
    current_liabilities_data = get_value_with_validation(
        us_gaap,
        ['LiabilitiesCurrent'],
        'current_liabilities'
    )
    
    total_debt_data = get_value_with_validation(
        us_gaap,
        ['LongTermDebt', 'DebtLongTerm'],
        'total_debt'
    )
    
    equity_data = get_value_with_validation(
        us_gaap,
        ['StockholdersEquity', 'ShareholdersEquity'],
        'shareholders_equity'
    )
    
    # Build result with metadata
    financial_data = {
        'ticker': ticker,
        'company_name': data.get('entityName'),
        'revenue': revenue_data['value'] if revenue_data else None,
        'revenue_date': revenue_data['date'] if revenue_data else None,
        'net_income': net_income_data['value'] if net_income_data else None,
        'total_assets': total_assets_data['value'] if total_assets_data else None,
        'current_assets': current_assets_data['value'] if current_assets_data else None,
        'current_liabilities': current_liabilities_data['value'] if current_liabilities_data else None,
        'total_debt': total_debt_data['value'] if total_debt_data else None,
        'shareholders_equity': equity_data['value'] if equity_data else None,
        
        # Metadata for verification
        'data_date': revenue_data['date'] if revenue_data else None,
        'fiscal_year': revenue_data['fiscal_year'] if revenue_data else None,
    }
    
    # Validation: Check if all dates match
    dates = [
        revenue_data['date'] if revenue_data else None,
        net_income_data['date'] if net_income_data else None,
        total_assets_data['date'] if total_assets_data else None,
    ]
    
    dates = [d for d in dates if d]  # Remove None values
    
    if len(set(dates)) > 1:
        print(f"⚠️  WARNING: Data from different periods detected!")
        print(f"   Dates found: {set(dates)}")
    
    print("Data Recived")
    return financial_data

# Test it
validated_data = get_financial_data_validated('MSFT')

✅ Found CIK: 0000789019
📡 Fetching data from SEC...
Data Recived


In [29]:
# Cell: Display validated data with metadata

if validated_data:
    print(f"\n{'='*60}")
    print(f"📊 VALIDATED FINANCIAL DATA: {validated_data['company_name']}")
    print(f"{'='*60}")
    print(f"📅 Data Date: {validated_data['data_date']}")
    print(f"📅 Fiscal Year: {validated_data['fiscal_year']}")
    print(f"{'='*60}\n")
    
    metrics = [
        ('Revenue', validated_data['revenue']),
        ('Net Income', validated_data['net_income']),
        ('Total Assets', validated_data['total_assets']),
        ('Current Assets', validated_data['current_assets']),
        ('Current Liabilities', validated_data['current_liabilities']),
        ('Total Debt', validated_data['total_debt']),
        ('Shareholders Equity', validated_data['shareholders_equity']),
    ]
    
    for name, value in metrics:
        if value is not None:
            print(f"{name:.<25} ${value:>20,}")
        else:
            print(f"{name:.<25} {'⚠️  NOT AVAILABLE':>20}")


📊 VALIDATED FINANCIAL DATA: MICROSOFT CORPORATION
📅 Data Date: 2025-06-30
📅 Fiscal Year: 2025

Revenue.................. $     281,724,000,000
Net Income............... $     101,832,000,000
Total Assets............. $     619,003,000,000
Current Assets........... $     191,131,000,000
Current Liabilities...... $     141,218,000,000
Total Debt............... $      43,151,000,000
Shareholders Equity...... $     343,479,000,000


In [24]:
data

{'ticker': 'MSFT',
 'company_name': 'MICROSOFT CORPORATION',
 'revenue': 281724000000,
 'net_income': 101832000000,
 'total_assets': 619003000000,
 'current_assets': 191131000000,
 'current_liabilities': 141218000000,
 'total_debt': 43151000000,
 'shareholders_equity': 343479000000}

In [23]:
if data:
    print(f"\n{'='*50}")
    print(f"FINANCIAL DATA: {data['company_name']}")
    print(f"{'='*50}\n")
    
    print(f"Revenue:              ${data['revenue']:>20,}")
    print(f"Net Income:           ${data['net_income']:>20,}")
    print(f"Total Assets:         ${data['total_assets']:>20,}")
    print(f"Current Assets:       ${data['current_assets']:>20,}")
    print(f"Current Liabilities:  ${data['current_liabilities']:>20,}")
    print(f"Total Debt:           ${data['total_debt']:>20,}")
    print(f"Shareholders Equity:  ${data['shareholders_equity']:>20,}")
else:
    print("Failed to get data")


FINANCIAL DATA: MICROSOFT CORPORATION

Revenue:              $     281,724,000,000
Net Income:           $     101,832,000,000
Total Assets:         $     619,003,000,000
Current Assets:       $     191,131,000,000
Current Liabilities:  $     141,218,000,000
Total Debt:           $      43,151,000,000
Shareholders Equity:  $     343,479,000,000
